# Figure 3 | Genetic regulation of novel lncRNA expression and splicing

**a.** Number of annotated genes versus genes with a significant eQTL (eGenes) at 5% FDR, comparing the GENCODE annotations used by GTEx v10 and v11 for lncRNAs and protein-coding genes. Points are colored by biotype; marker style indicates annotation version (v10 vs v11), and lines connect the same biotype across versions. Updated annotation yields 9,354 additional lncRNA eGenes relative to v10.

**b.** Number of SuSiE fine-mapping eQTL credible sets per tissue for novel lncRNAs (filled) and existing lncRNAs (hollow). Tissues are ordered by novel lncRNA eQTL credible-set count. Novel lncRNAs contribute a median of 682 eQTL credible sets per tissue.

**c.** Same as (a) for sQTLs / sGenes. Updated annotation yields 2,502 additional lncRNA sGenes relative to v10.

**d.** Same as (b) for sQTL credible sets (square markers). Novel lncRNAs contribute a median of 53 sQTL credible sets per tissue.

**e.** Fold enrichment of lead fine-mapped QTL variants (posterior inclusion probability, PIP ≥ 0.9) within selected functional genomic annotations, shown for novel lncRNAs (filled) and existing lncRNAs (hollow) across eQTL (circles) and sQTL (squares) credible sets. Annotations: lncRNA splicing, protein-coding (p.c.) splicing, lncRNA exon, p.c. exon, promoter, lncRNA intron, CCCTC-binding factor (CTCF) binding sites, and enhancer. Error bars, 95% confidence intervals (CIs). Annotations are ordered by enrichment in novel lncRNA eQTLs. Novel and existing lncRNA QTLs are enriched in similar genomic features.

**f.** Tissue-weighted area under the receiver-operating characteristic curve (auROC) for distinguishing fine-mapped eQTL variants from distance-matched controls on the protein-coding holdout, as a function of AlphaGenome random-forest (RF) training set size. Curves show models trained on protein-coding, novel lncRNA, or existing lncRNA labels; the dotted line indicates zero-shot AlphaGenome performance. Points, mean auROC across chromosome folds; error bars, fold-level 95% CIs. Models trained on lncRNA labels reach predictive performance comparable to protein-coding-trained models on the protein-coding holdout, indicating shared sequence features of fine-mapped eQTL variants across biotypes.


## Setup


In [ ]:
import glob
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from tqdm.notebook import tqdm

plt.rc("axes", labelsize=6)
plt.rc("xtick", labelsize=6)
plt.rc("ytick", labelsize=6)
plt.rc("legend", fontsize=5)
plt.rc("legend", title_fontsize=6)
plt.rc("font", size=6)
plt.rc("figure", titlesize=7)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["figure.figsize"] = (7.2, 5.4)
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300

# Plot / analysis parameters
fig_width, fig_height, fig_dpi = 7.2, 5.4, 300
qval_fdr = 0.05
annot_version = "v11"
pip_cutoff = 0.9
label_subsample_grid = [50, 100, 500, 1000, 5000, 10000, 50000]
label_subsample_train_biotypes = ["protein_coding", "novel_lncRNA", "existing_lncRNA"]
label_subsample_panel_eval = "protein_coding"

gene_type_pretty = {
    "novel_lncRNA": "novel lncRNA",
    "existing_lncRNA": "existing lncRNA",
    "protein_coding": "protein-coding",
}
_new_lnc = sns.color_palette("Paired", 12)[5]
_exist_lnc = sns.color_palette("Paired", 12)[4]
gene_type_palette = {
    "lncRNA": tuple(0.5 * (a + b) for a, b in zip(_new_lnc, _exist_lnc)),
    "novel_lncRNA": _new_lnc,
    "existing_lncRNA": _exist_lnc,
    "protein_coding": sns.color_palette("Paired", 12)[1],
    "pseudogene": sns.color_palette("Paired", 12)[3],
}
gene_type_count_plot = ["lncRNA", "protein_coding"]

enrichment_annotations = [
    "splicing_merged_lncRNA_d", "splicing_merged_protein_coding_d",
    "non_coding_transcript_exon_variant_lncRNA_d", "non_coding_transcript_exon_variant_protein_coding_d",
    "promoter_d", "intron_variant_lncRNA_d", "CTCF_binding_site_d", "enhancer_d",
]
pretty_annotations = {
    "splicing_merged_lncRNA_d": "lncRNA\nsplicing",
    "splicing_merged_protein_coding_d": "p.c.\nsplicing",
    "non_coding_transcript_exon_variant_lncRNA_d": "lncRNA\nexon",
    "non_coding_transcript_exon_variant_protein_coding_d": "p.c.\nexon",
    "promoter_d": "promoter",
    "intron_variant_lncRNA_d": "lncRNA\nintron",
    "CTCF_binding_site_d": "CTCF",
    "enhancer_d": "enhancer",
}

## Paths

In [ ]:
gtex_colors_file = Path("refs/colors.json")
qtl_tissue_ids_file = Path("path/to/qtl_tissue_ids.txt")
gencode_class_file = Path("path/to/gencode_v39_v47_gene_classification.txt")
eqtl_dir = Path("path/to/v11_eqtl_susie")
sqtl_dir = Path("path/to/v11_sqtl_susie")
v10_egene_dir = Path("path/to/v10_eqtl_susie")
v10_sgene_dir = Path("path/to/v10_sqtl_susie")
odds_file = Path("path/to/qtl_vep_odds.txt")

# AlphaGenome QTL outputs (panel f): from step 04 (matched-N curves + zero-shot rows)
learning_curves_file = Path("path/to/qtl_causality_auroc.tsv")

fig_dir = Path("figures")
fig_pdf = fig_dir / "figure_3.pdf"
fig_png = fig_dir / "figure_3.png"


## Load data


### Tissue colors

In [ ]:
with open(gtex_colors_file) as f:
    _colors = json.load(f)
    tissue_df = pd.DataFrame.from_dict(_colors, orient="index")
tissue_df = tissue_df.reset_index(names="tissue_name")
tissue_df["tissue_file_name"] = (
    tissue_df["tissue_name"]
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace(" - ", "_")
    .str.replace(" ", "_")
)


### QTL tissue set

In [ ]:
qtl_tissue_set = set(pd.read_table(qtl_tissue_ids_file, header=None)[0])
tissue_df["qtl_tissue"] = tissue_df["tissue_file_name"].isin(qtl_tissue_set)
tissue_df_indexed = tissue_df.set_index("tissue_file_name")
qtl_tissues = tissue_df.loc[tissue_df["qtl_tissue"], "tissue_file_name"]
if len(qtl_tissues) != len(qtl_tissue_set):
    missing = sorted(qtl_tissue_set - set(qtl_tissues))
    raise ValueError(
        f"Expected {len(qtl_tissue_set)} QTL tissues in tissue_df, got {len(qtl_tissues)}; missing: {missing}"
    )

### GENCODE gene classification

In [ ]:
gencode_genes = pd.read_table(gencode_class_file, low_memory=False)
gencode_genes = gencode_genes[gencode_genes["gene_type_grouped"] != "artifact"]
new_lncrna_ids = set(gencode_genes.loc[gencode_genes["gene_type_grouped"] == "novel_lncRNA", "gene_id_v47"])
existing_lncrna_ids = set(gencode_genes.loc[gencode_genes["gene_type_grouped"] == "existing_lncRNA", "gene_id_v47"])

print(f"QTL tissues: {len(qtl_tissues)}; gencode genes: {len(gencode_genes):,}")

### v11 e/sGenes by tissue (FDR 5%)


In [ ]:
egene_parts, sgene_parts = [], []
for tissue_id in tqdm(qtl_tissues, desc="e/sGenes"):
    egene_path = eqtl_dir / f"{tissue_id}.v11.eGenes.txt.gz"
    sgene_path = sqtl_dir / f"{tissue_id}.v11.sGenes.txt.gz"
    tissue_egenes = pd.read_table(egene_path, usecols=["gene_id", "qval"])
    tissue_egenes[f"{tissue_id}.is_egene"] = tissue_egenes["qval"] < qval_fdr
    egene_parts.append(tissue_egenes[["gene_id", f"{tissue_id}.is_egene"]].set_index("gene_id"))

    tissue_sgenes = pd.read_table(sgene_path, usecols=["gene_id", "qval"])
    tissue_sgenes[f"{tissue_id}.is_sgene"] = tissue_sgenes["qval"] < qval_fdr
    sgene_parts.append(tissue_sgenes[["gene_id", f"{tissue_id}.is_sgene"]].set_index("gene_id"))

egenes = pd.concat(egene_parts, axis=1)
sgenes = pd.concat(sgene_parts, axis=1)
qtl_genes = pd.merge(egenes, sgenes, left_index=True, right_index=True, how="outer")
qtl_genes = pd.merge(
    qtl_genes,
    gencode_genes[~gencode_genes["gene_id_v47"].isna()],
    left_index=True,
    right_on="gene_id_v47",
    how="right",
)

egene_columns = [f"{t}.is_egene" for t in qtl_tissues]
sgene_columns = [f"{t}.is_sgene" for t in qtl_tissues]
qtl_genes["num_egenes"] = qtl_genes[egene_columns].fillna(False).sum(axis=1)
qtl_genes["num_sgenes"] = qtl_genes[sgene_columns].fillna(False).sum(axis=1)
print(f"QTL gene table: {len(qtl_genes):,} genes")

### v10 e/sGenes (annotated-gene scatters)


In [ ]:
egenes_v10_parts, sgenes_v10_parts = [], []
for tissue_id in tqdm(qtl_tissues, desc="v10 e/sGenes"):
    egene_path = v10_egene_dir / f"{tissue_id}.v10.eGenes.txt.gz"
    sgene_path = v10_sgene_dir / f"{tissue_id}.v10.sGenes.txt.gz"
    tissue_egenes = pd.read_table(egene_path, usecols=["gene_id", "qval"])
    tissue_egenes[f"{tissue_id}.is_egene"] = tissue_egenes["qval"] < qval_fdr
    egenes_v10_parts.append(tissue_egenes[["gene_id", f"{tissue_id}.is_egene"]].set_index("gene_id"))

    tissue_sgenes = pd.read_table(sgene_path, usecols=["gene_id", "qval"])
    tissue_sgenes[f"{tissue_id}.is_sgene"] = tissue_sgenes["qval"] < qval_fdr
    sgenes_v10_parts.append(tissue_sgenes[["gene_id", f"{tissue_id}.is_sgene"]].set_index("gene_id"))

egenes_v10 = pd.concat(egenes_v10_parts, axis=1)
sgenes_v10 = pd.concat(sgenes_v10_parts, axis=1)
qtl_genes_v10 = pd.merge(egenes_v10, sgenes_v10, left_index=True, right_index=True, how="outer")
qtl_genes_v10 = pd.merge(
    qtl_genes_v10,
    gencode_genes[~gencode_genes["gene_id_v39"].isna()],
    left_index=True,
    right_on="gene_id_v39",
    how="right",
)

egene_columns_v10 = [f"{t}.is_egene" for t in qtl_tissues]
sgene_columns_v10 = [f"{t}.is_sgene" for t in qtl_tissues]
qtl_genes_v10["num_egenes"] = qtl_genes_v10[egene_columns_v10].fillna(False).sum(axis=1)
qtl_genes_v10["num_sgenes"] = qtl_genes_v10[sgene_columns_v10].fillna(False).sum(axis=1)
print(f"v10 QTL gene table: {len(qtl_genes_v10):,} genes ({len(egene_columns_v10)} tissues)")

### SuSiE summaries


In [ ]:
susie_e = {}
for tissue_id in tqdm(qtl_tissues, desc="eQTLs"):
    path = eqtl_dir / f"{tissue_id}.v11.eQTLs.SuSiE_summary.parquet"
    df = pd.read_parquet(path)
    df["gene_id"] = df["phenotype_id"].str.split(":").str[-1]
    susie_e[tissue_id] = df

susie_s = {}
for tissue_id in tqdm(qtl_tissues, desc="sQTLs"):
    path = sqtl_dir / f"{tissue_id}.v11.sQTLs.SuSiE_summary.parquet"
    df = pd.read_parquet(path)
    df["gene_id"] = df["phenotype_id"].str.split(":").str[-1]
    susie_s[tissue_id] = df
print(f"SuSiE summaries loaded for {len(susie_e)} tissues")

### Variant annotation odds


In [ ]:
print(f"Loading {odds_file.name} ({annot_version}; panel e PIP>{pip_cutoff})")
odds = pd.read_table(odds_file)
odds["annotation"] = odds["annotation"].str.replace(r"_v1[01]$", "", regex=True)
if pip_cutoff not in set(odds["pip_cutoff"].dropna()):
    raise ValueError(
        f"pip_cutoff={pip_cutoff} missing from odds output (have {sorted(odds['pip_cutoff'].dropna().unique())})"
    )


### AlphaGenome label-subsample curves

In [ ]:
learning_curves = pd.read_csv(learning_curves_file, sep="\t")
print(f"Learning curves: {len(learning_curves):,} rows")

## Run analysis


### Credible-set counts


In [ ]:
cs_rows_e, cs_rows_s = [], []
for tissue_id, df in susie_e.items():
    cs = df.drop_duplicates(["phenotype_id", "cs_id"])
    cs_rows_e.append({
        "tissue_id": tissue_id,
        "new_lncrna_e_counts": cs["gene_id"].isin(new_lncrna_ids).sum(),
        "existing_lncrna_e_counts": cs["gene_id"].isin(existing_lncrna_ids).sum(),
    })
for tissue_id, df in susie_s.items():
    cs = df.drop_duplicates(["phenotype_id", "cs_id"])
    cs_rows_s.append({
        "tissue_id": tissue_id,
        "new_lncrna_s_counts": cs["gene_id"].isin(new_lncrna_ids).sum(),
        "existing_lncrna_s_counts": cs["gene_id"].isin(existing_lncrna_ids).sum(),
    })

susie_counts_e = pd.DataFrame(cs_rows_e)
susie_counts_s = pd.DataFrame(cs_rows_s)
susie_counts_e["tissue_abbrv"] = susie_counts_e["tissue_id"].map(tissue_df_indexed["tissue_abbrv"])
susie_counts_s["tissue_abbrv"] = susie_counts_s["tissue_id"].map(tissue_df_indexed["tissue_abbrv"])
susie_counts_e = susie_counts_e.sort_values("new_lncrna_e_counts", ascending=False).reset_index(drop=True)
susie_counts_s = susie_counts_s.sort_values("new_lncrna_s_counts", ascending=False).reset_index(drop=True)
new_e_median = np.median(susie_counts_e["new_lncrna_e_counts"])
new_s_median = np.median(susie_counts_s["new_lncrna_s_counts"])
print(f"Median novel lncRNA CS/tissue: eQTL={new_e_median:.0f}, sQTL={new_s_median:.0f}")

### Variant enrichment tables


In [ ]:
df_e = odds[
    (odds["pip_cutoff"] == pip_cutoff) & (odds["annot_version"] == annot_version)
    & (odds["susie_type"] == "eqtl") & (odds["gene_type"].isin(["novel_lncRNA", "existing_lncRNA"]))
    & odds["annotation"].isin(enrichment_annotations)
].dropna(subset=["odds_ratio", "ci_lower", "ci_upper"], how="all").copy()

df_s = odds[
    (odds["pip_cutoff"] == pip_cutoff) & (odds["annot_version"] == annot_version)
    & (odds["susie_type"] == "sqtl") & (odds["gene_type"].isin(["novel_lncRNA", "existing_lncRNA"]))
    & odds["annotation"].isin(enrichment_annotations)
].dropna(subset=["odds_ratio", "ci_lower", "ci_upper"], how="all").copy()

annot_order_e = (
    df_e[df_e["gene_type"] == "novel_lncRNA"].groupby("annotation")["odds_ratio"].max().sort_values(ascending=False).index.tolist()
)
annot_order_s = (
    df_s[df_s["gene_type"] == "novel_lncRNA"].groupby("annotation")["odds_ratio"].max().sort_values(ascending=False).index.tolist()
)
annot_order = annot_order_e if len(annot_order_e) >= len(annot_order_s) else annot_order_s
if not annot_order:
    raise ValueError("No enrichment annotations available after filtering for panel e.")
print(f"Enrichment annotations: {len(annot_order)}; eQTL rows={len(df_e)}, sQTL rows={len(df_s)}")

### AlphaGenome zero-shot baseline and matched-N curves (panel f)

In [ ]:
panel_f_all = learning_curves[
    learning_curves["eval_biotype"] == label_subsample_panel_eval
].copy()
# Matched-N curves are v11; exclude catalog rf_v10/rf_v11 rows. Prefer v11 zero-shot.
if "dataset_version" in panel_f_all.columns:
    panel_f_all = panel_f_all[
        panel_f_all["dataset_version"].isna()
        | (panel_f_all["dataset_version"].astype(str) == "v11")
    ].copy()
panel_f_all = panel_f_all[
    ~panel_f_all["train_biotype"].isin(["rf_v10", "rf_v11"])
].copy()
panel_f_curves = panel_f_all[panel_f_all["train_biotype"] != "zero_shot"].copy()
zero_shot_row = panel_f_all.loc[panel_f_all["train_biotype"] == "zero_shot"]
if panel_f_curves.empty:
    raise FileNotFoundError(
        f"No RF curve rows for eval={label_subsample_panel_eval} in {learning_curves_file}"
    )
if zero_shot_row.empty:
    raise FileNotFoundError(
        f"No zero_shot row for eval={label_subsample_panel_eval} in {learning_curves_file}"
    )
panel_f_zero_shot = (float(zero_shot_row["mean"].iloc[0]),)
print(
    f"Panel f eval: {label_subsample_panel_eval}; "
    f"curve rows={len(panel_f_curves)}; zero-shot {panel_f_zero_shot[0]:.4f}"
)

## Make figure


In [ ]:
fig = plt.figure(figsize=(fig_width, fig_height), dpi=fig_dpi)

# Shared columns so a/c and b/d align; left column = 1/4, right = 3/4.
# Bottom row e/f uses nested widths (3:1) so enrichment stays wide on the left.
gs = GridSpec(
    3, 2, figure=fig,
    height_ratios=[0.28, 0.28, 0.55],
    width_ratios=[1, 3],
    hspace=0.65,
    wspace=0.28,
    left=0.10,
    right=0.98,
    top=0.94,
    bottom=0.12,
)
ax_e_n = fig.add_subplot(gs[0, 0])
ax_cs_e = fig.add_subplot(gs[0, 1])
ax_s_n = fig.add_subplot(gs[1, 0])
ax_cs_s = fig.add_subplot(gs[1, 1])
gs_ef = gs[2, :].subgridspec(1, 2, width_ratios=[3, 1], wspace=0.28)
ax_enr = fig.add_subplot(gs_ef[0, 0])  # e: enrichment (left, wide)
ax_ag = fig.add_subplot(gs_ef[0, 1])   # f: AlphaGenome (right, narrow)

ref_vline_kw = dict(color="grey", linestyle="--", linewidth=1, alpha=0.6, zorder=0)
new_color = gene_type_palette["novel_lncRNA"]
exist_color = gene_type_palette["existing_lncRNA"]

# Panels a/c: annotated genes → e/sGenes
version_legend = [
    Line2D([0], [0], marker="1", color="k", label="v10", markerfacecolor="grey",
           markeredgecolor="k", markersize=5, linestyle="None", linewidth=1),
    Line2D([0], [0], marker="2", color="k", label="v11", markerfacecolor="grey",
           markeredgecolor="k", markersize=5, linestyle="None", linewidth=1),
]
for ax, ycol, ylab in [
    (ax_e_n, "num_egenes", "eGenes\n(FDR 5%)"),
    (ax_s_n, "num_sgenes", "sGenes\n(FDR 5%)"),
]:
    for gtype in gene_type_count_plot:
        # gene_type_grouped stores novel/existing lncRNA split; collapse for this panel
        gtypes = ["novel_lncRNA", "existing_lncRNA"] if gtype == "lncRNA" else [gtype]
        n_v10 = (gencode_genes["gene_type_grouped"].isin(gtypes) & gencode_genes["gene_id_v39"].notna()).sum()
        n_v11 = (gencode_genes["gene_type_grouped"].isin(gtypes) & gencode_genes["gene_id_v47"].notna()).sum()
        y_v10 = (qtl_genes_v10["gene_type_grouped"].isin(gtypes) & (qtl_genes_v10[ycol] > 0)).sum()
        y_v11 = (qtl_genes["gene_type_grouped"].isin(gtypes) & (qtl_genes[ycol] > 0)).sum()
        color = gene_type_palette[gtype]
        ax.scatter(n_v10, y_v10, color=color, marker="1", s=48, edgecolor="k", linewidths=1.2, zorder=4)
        ax.scatter(n_v11, y_v11, color=color, marker="2", s=48, edgecolor="k", linewidths=1.2, zorder=3)
        ax.plot([n_v10, n_v11], [y_v10, y_v11], color=color, linewidth=0.8, zorder=2)
    ax.set_xlabel("Annotated genes")
    ax.set_ylabel(ylab)
    ax.set_xticks([0, 10000, 20000, 30000])
    ax.set_xticklabels(["0", "10k", "20k", "30k"])
    ax.set_yticks([0, 10000, 20000])
    ax.set_yticklabels(["0", "10k", "20k"])
    ax.set_xlim(0, 38000)
    ax.set_ylim(-1000, 26000)
    sns.despine(ax=ax)

for ax in [ax_e_n, ax_s_n]:
    ax.legend(
        handles=version_legend,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.22),
        frameon=False,
        ncol=2,
        fontsize=6,
        handletextpad=0.2,
        columnspacing=0.8,
    )

counts_base = susie_counts_e.merge(
    susie_counts_s[["tissue_id", "new_lncrna_s_counts", "existing_lncrna_s_counts"]],
    on="tissue_id",
).sort_values("new_lncrna_e_counts", ascending=False).reset_index(drop=True)
tissue_order = counts_base["tissue_id"].tolist()


def plot_cs_panel(ax, counts_df, new_col, exist_col, marker, ylabel):
    counts_df = counts_df.set_index("tissue_id").loc[tissue_order].reset_index()
    xticks = np.arange(len(counts_df))
    for i in range(len(counts_df)):
        if i % 2 == 1:
            ax.axvspan(i - 0.5, i + 0.5, color="#eeeeee", zorder=0)
    ax.scatter(
        xticks - 0.04,
        counts_df[exist_col],
        marker=marker,
        s=6,
        c="white",
        edgecolors=exist_color,
        linewidths=0.8,
        alpha=0.9,
        zorder=1,
    )
    ax.scatter(
        xticks + 0.04,
        counts_df[new_col],
        marker=marker,
        s=6,
        c=new_color,
        edgecolors=new_color,
        linewidths=0.8,
        alpha=0.9,
        zorder=2,
    )
    ax.set_ylabel(ylabel)
    ax.set_xticks(xticks)
    ax.set_xticklabels(counts_df["tissue_abbrv"], fontsize=4, rotation=90, ha="center")
    ax.set_xlim(-0.5, len(counts_df) - 0.5)
    ax.set_yscale("log")
    ax.set_yticks([10, 100, 1000, 10000])
    ax.tick_params(axis="x", length=0)
    sns.despine(ax=ax, bottom=True)

# Prepare legend handles for credible set plots
cs_handles_eqtl = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=new_color, markeredgecolor=new_color, markersize=2.5, linestyle="None", label=gene_type_pretty["novel_lncRNA"]),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="w", markeredgecolor=exist_color, markersize=2.5, linestyle="None", label=gene_type_pretty["existing_lncRNA"]),
]
cs_handles_sqtl = [
    Line2D([0], [0], marker="s", color="w", markerfacecolor=new_color, markeredgecolor=new_color, markersize=2.5, linestyle="None", label=gene_type_pretty["novel_lncRNA"]),
    Line2D([0], [0], marker="s", color="w", markerfacecolor="w", markeredgecolor=exist_color, markersize=2.5, linestyle="None", label=gene_type_pretty["existing_lncRNA"]),
]

plot_cs_panel(
    ax_cs_e, counts_base, "new_lncrna_e_counts", "existing_lncrna_e_counts",
    "o", "eQTL credible sets",
)
plot_cs_panel(
    ax_cs_s, counts_base, "new_lncrna_s_counts", "existing_lncrna_s_counts",
    "s", "sQTL credible sets",
)

# Add legends for credible sets panels (centered over the plots)
ax_cs_e.legend(
    handles=cs_handles_eqtl,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.26),
    frameon=False,
    ncol=2,
    fontsize=5.2,
    handletextpad=0.4,
    columnspacing=1.0,
    borderaxespad=0.0,
)
ax_cs_s.legend(
    handles=cs_handles_sqtl,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.26),
    frameon=False,
    ncol=2,
    fontsize=5.2,
    handletextpad=0.4,
    columnspacing=1.0,
    borderaxespad=0.0,
)

# Panel e: variant enrichment (categories on x, enrichment on y)
xmap = {ann: i for i, ann in enumerate(annot_order)}
for i in range(len(annot_order)):
    if i % 2 == 1:
        ax_enr.axvspan(i - 0.5, i + 0.5, color="#eeeeee", zorder=0)
ax_enr.axhline(0, **ref_vline_kw)
ax_enr.set_xticks(range(len(annot_order)))
ax_enr.tick_params(axis="x", length=0, pad=1)
ax_enr.set_xticklabels(
    [pretty_annotations.get(a, a) for a in annot_order],
    linespacing=0.95,
    fontsize=5.5,
)
# offsets: eQTL novel / eQTL existing / sQTL novel / sQTL existing
_enr_offsets = {
    ("eqtl", "novel_lncRNA"): -0.24,
    ("eqtl", "existing_lncRNA"): -0.08,
    ("sqtl", "novel_lncRNA"): 0.08,
    ("sqtl", "existing_lncRNA"): 0.24,
}
for susie_type, df, marker in [("eqtl", df_e, "o"), ("sqtl", df_s, "s")]:
    for gene_type in ["novel_lncRNA", "existing_lncRNA"]:
        color = gene_type_palette[gene_type]
        d = df[(df["gene_type"] == gene_type) & df["annotation"].isin(annot_order)]
        if d.empty:
            continue
        x = d["annotation"].map(xmap).to_numpy() + _enr_offsets[(susie_type, gene_type)]
        y = np.log2(d["odds_ratio"].to_numpy())
        ax_enr.errorbar(
            x,
            y,
            yerr=[y - np.log2(d["ci_lower"]), np.log2(d["ci_upper"]) - y],
            color=color,
            marker=marker,
            markerfacecolor=color,
            markeredgecolor="k",
            markeredgewidth=0.5,
            ecolor=color,
            capsize=0,
            markersize=3.5,
            linestyle="none",
            elinewidth=0.8,
        )
ax_enr.set_ylabel("Fold enrichment in e/sVars")
ax_enr.set_yticks([-4, -2, 0, 2, 4, 6, 8])
ax_enr.set_yticklabels(["2$^{-4}$", "2$^{-2}$", "2$^{0}$", "2$^{2}$", "2$^{4}$", "2$^{6}$", "2$^{8}$"])
ax_enr.set_xlim(-0.6, len(annot_order) - 0.4)
sns.despine(ax=ax_enr, bottom=True)

enr_handles = [
    Line2D([0], [0], marker="o", color="k", markerfacecolor="grey", markeredgecolor="k",
           markeredgewidth=0.5, linestyle="None", label="eVariant", markersize=3.5),
    Line2D([0], [0], marker="o", color="k", markerfacecolor=new_color, markeredgecolor="k",
           markeredgewidth=0.5, linestyle="None", label=gene_type_pretty["novel_lncRNA"], markersize=3.5),
    Line2D([0], [0], marker="s", color="k", markerfacecolor="grey", markeredgecolor="k",
           markeredgewidth=0.5, linestyle="None", label="sVariant", markersize=3.5),
    Line2D([0], [0], marker="o", color="k", markerfacecolor=exist_color, markeredgecolor="k",
           markeredgewidth=0.5, linestyle="None", label=gene_type_pretty["existing_lncRNA"], markersize=3.5),
]
ax_enr.legend(
    handles=enr_handles,
    loc="lower center",
    frameon=False,
    ncol=2,
    bbox_to_anchor=(0.5, 0.96),
    fontsize=5.2,
    columnspacing=1.0,
    handletextpad=0.4,
)

# Panel f: matched-N learning curve on protein-coding holdout
x_max = label_subsample_grid[-1]
for train_bt in label_subsample_train_biotypes:
    curve = (
        panel_f_curves[panel_f_curves["train_biotype"] == train_bt]
        .sort_values("n_train")
        .copy()
    )
    if curve.empty:
        continue
    color = gene_type_palette[train_bt]
    yerr_lo = curve["mean"] - curve["ci_lower"]
    yerr_hi = curve["ci_upper"] - curve["mean"]
    ax_ag.errorbar(
        curve["n_train"],
        curve["mean"],
        yerr=[yerr_lo, yerr_hi],
        fmt="-o",
        color=color,
        ecolor=color,
        elinewidth=0.8,
        capsize=0,
        markersize=3.0,
        markeredgecolor="k",
        markeredgewidth=0.35,
        linewidth=0.8,
        zorder=3,
        label=gene_type_pretty[train_bt],
    )
    x_max = max(x_max, int(curve["n_train"].max()))

ax_ag.axhline(
    panel_f_zero_shot[0],
    color="#666666",
    linestyle=":",
    linewidth=0.9,
    alpha=0.9,
    zorder=1,
)
ax_ag.text(
    0.98,
    panel_f_zero_shot[0],
    "zero-shot",
    transform=ax_ag.get_yaxis_transform(),
    ha="right",
    va="bottom",
    fontsize=5,
    color="#666666",
    clip_on=False,
)
ax_ag.set_xscale("log")
ax_ag.set_xlim(label_subsample_grid[0] * 0.8, x_max * 1.15)
ax_ag.set_xticks([100, 1000, 10000, 50000])
ax_ag.set_xticklabels(["100", "1k", "10k", "50k"])
ax_ag.set_xlabel("AlphaGenome RF training size")
ax_ag.set_ylabel("tissue-weighted auROC\n(evaluated on protein-coding)")
ax_ag.set_ylim(0.5, 0.82)
ax_ag.set_yticks([0.5, 0.6, 0.7, 0.8])
ax_ag.legend(
    frameon=False,
    loc="lower right",
    fontsize=4.5,
    title="Training biotype",
)
sns.despine(ax=ax_ag)

fig.canvas.draw()

# Panel labels: a|b → c|d → e|f (shared left-edge x for a/c; e uses enr left)
_inv = fig.transFigure.inverted()
_left_x = _inv.transform(ax_e_n.transAxes.transform((-0.22, 0)))[0]
for ax, label, y_ax, use_left in [
    (ax_e_n, "a", 1.12, True), (ax_cs_e, "b", 1.12, False),
    (ax_s_n, "c", 1.12, True), (ax_cs_s, "d", 1.12, False),
    (ax_enr, "e", 1.04, True), (ax_ag, "f", 1.04, False),
]:
    if use_left:
        _x = _left_x
    else:
        _x = _inv.transform(ax.transAxes.transform((-0.08, 0)))[0]
    _y = _inv.transform(ax.transAxes.transform((0, y_ax)))[1]
    fig.text(_x, _y, label, fontweight="bold", va="bottom", ha="right", fontsize=9)
plt.show()

## Save figure


In [ ]:
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_pdf, bbox_inches="tight")
fig.savefig(fig_png, bbox_inches="tight")
print(f"Wrote {fig_pdf}")
print(f"Wrote {fig_png}")

## Print stats


In [ ]:
# Figure 3 Results stats
print(f"QTL tissues (sample size > 70): {len(susie_e)}")

for ycol, label in [("num_egenes", "eGenes"), ("num_sgenes", "sGenes")]:
    n_v10 = int(((qtl_genes_v10["gene_type_grouped"].isin(["novel_lncRNA", "existing_lncRNA"])) & (qtl_genes_v10[ycol] > 0)).sum())
    n_v11 = int(((qtl_genes["gene_type_grouped"].isin(["novel_lncRNA", "existing_lncRNA"])) & (qtl_genes[ycol] > 0)).sum())
    print(f"additional lncRNA {label}: {n_v11 - n_v10:,}")

for split, label in [("novel_lncRNA", "novel"), ("existing_lncRNA", "existing")]:
    sub = qtl_genes[qtl_genes["gene_type_grouped"] == split]
    for ycol, qlabel in [("num_egenes", "eGenes"), ("num_sgenes", "sGenes")]:
        k = int((sub[ycol] > 0).sum())
        print(f"{label} lncRNA {qlabel}: {k:,}/{len(sub):,} ({100 * k / len(sub):.0f}%)")

print(f"median novel eQTL CS/tissue: {new_e_median:.0f}")
print(f"median novel sQTL CS/tissue: {new_s_median:.0f}")
